Absolutely. Now we’ll combine **Parts 8–13** into one practical XGBoost section:

**XGBClassifier → XGBRegressor → Evaluation → Hyperparameter Tuning → Early Stopping → Feature Importance**

This is the part where you move from understanding XGBoost theoretically to actually using it on datasets.

# XGBoost — Practical Implementation & Model Tuning

---

# 1. Installing XGBoost

If you haven't installed it:

```bash
pip install xgboost
```

Check the installation:

```python
import xgboost

print(xgboost.__version__)
```

Import the models:

```python
from xgboost import XGBClassifier
from xgboost import XGBRegressor
```

---

# 2. XGBClassifier

`XGBClassifier` is used when your target variable is categorical.

Examples:

```text
Spam / Not Spam
Disease / No Disease
0 / 1
Yes / No
```

Basic structure:

```python
from xgboost import XGBClassifier

model = XGBClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
```

---

# 3. Binary Classification

For binary classification, a common objective is:

```python
objective="binary:logistic"
```

This uses logistic loss and produces probabilities between:

$$
0 \leq P(y=1|x) \leq 1
$$

Example:

```python
model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
```

---

# 4. Complete XGBClassifier Example

Let's use the Breast Cancer dataset.

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))
```

---

# 5. `predict()` vs `predict_proba()`

This is important.

### `predict()`

Returns the final class.

```python
y_pred = model.predict(X_test)
```

Output:

```text
[0 1 1 0 1 ...]
```

---

### `predict_proba()`

Returns probabilities for each class.

```python
y_prob = model.predict_proba(X_test)
```

Example:

```text
[[0.90, 0.10],
 [0.15, 0.85],
 [0.20, 0.80]]
```

The columns represent:

```text
Column 0 → P(class 0)
Column 1 → P(class 1)
```

Therefore:

```python
y_prob = model.predict_proba(X_test)[:, 1]
```

gives:

```text
P(class = 1)
```

---

# 6. Classification Evaluation

Don't rely only on accuracy.

You can use:

### Accuracy

$$
Accuracy=
\frac{TP+TN}{TP+TN+FP+FN}
$$

```python
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)
```

---

### Precision

$$
Precision=
\frac{TP}{TP+FP}
$$

```python
from sklearn.metrics import precision_score

precision_score(y_test, y_pred)
```

---

### Recall

$$
Recall=
\frac{TP}{TP+FN}
$$

```python
from sklearn.metrics import recall_score

recall_score(y_test, y_pred)
```

---

### F1 Score

$$
F1=
2\frac{Precision\times Recall}
{Precision+Recall}
$$

```python
from sklearn.metrics import f1_score

f1_score(y_test, y_pred)
```

---

# 7. Confusion Matrix

```python
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)
```

You can visualize it:

```python
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.show()
```

---

# 8. ROC-AUC

Since XGBoost can produce probabilities:

```python
y_prob = model.predict_proba(X_test)[:, 1]
```

we can calculate ROC-AUC:

```python
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)
```

Important:

> ROC-AUC should generally be calculated using probabilities/scores, not hard class predictions.

---

# 9. XGBRegressor

Now regression.

Use:

```python
XGBRegressor
```

when the target is continuous.

Examples:

```text
House price
Salary
Temperature
Sales
Revenue
```

Basic structure:

```python
from xgboost import XGBRegressor

model = XGBRegressor()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
```

---

# 10. Regression Objective

A common regression objective is:

```python
objective="reg:squarederror"
```

This uses squared error.

Conceptually:

$$
L =
\frac{1}{2}(y-\hat y)^2
$$

The model tries to minimize the prediction error while also considering regularization.

---

# 11. California Housing Example

Since you've already worked with California Housing using:

* Decision Tree
* AdaBoost
* Gradient Boosting

this is an excellent dataset for comparing XGBoost against your previous models.

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
```

Create the model:

```python
from xgboost import XGBRegressor

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
```

---

# 12. Regression Evaluation

### R² Score

$$
R^2 =
1-
\frac{SS_{res}}{SS_{tot}}
$$

```python
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("R²:", r2)
```

---

### Mean Squared Error

$$
MSE =
\frac{1}{n}
\sum(y_i-\hat y_i)^2
$$

```python
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print("MSE:", mse)
```

---

### Root Mean Squared Error

$$
RMSE=\sqrt{MSE}
$$

Depending on your installed scikit-learn version, you can use:

```python
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)

print("RMSE:", rmse)
```

Or calculate it directly:

```python
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)
```

---

# 13. Complete Regression Evaluation

```python
from sklearn.metrics import (
    r2_score,
    mean_squared_error
)

import numpy as np

print("R²:",
      r2_score(y_test, y_pred))

print("MSE:",
      mean_squared_error(y_test, y_pred))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, y_pred)))
```

---

# 14. Do We Need Feature Scaling?

Usually:

```text
XGBoost → NO StandardScaler required
```

Because XGBoost uses decision trees.

For example:

```text
Age < 30
```

is essentially unaffected by monotonic scaling of the feature.

Therefore you normally don't need:

```python
from sklearn.preprocessing import StandardScaler
```

for XGBoost.

This is different from models such as:

* KNN
* Logistic Regression
* SVM
* Neural Networks

where scaling can be important.

---

# 15. Hyperparameter Tuning

Now we get to the important part.

You already learned:

```python
GridSearchCV
RandomizedSearchCV
```

The same concepts apply to XGBoost.

---

# 16. GridSearchCV

Grid search tests every combination.

Example:

```python
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
```

---

# 17. Why GridSearch Can Become Expensive

Suppose you have:

```text
n_estimators → 4
learning_rate → 4
max_depth → 5
subsample → 3
colsample_bytree → 3
```

Total combinations:

$$
4\times4\times5\times3\times3
$$

$$
=720
$$

With:

```python
cv=5
```

you perform:

$$
720\times5=3600
$$

model fits.

That's expensive.

And XGBoost itself can involve hundreds of trees.

---

# 18. RandomizedSearchCV

This is often more practical when you have many parameters.

```python
from sklearn.model_selection import RandomizedSearchCV

params = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 2, 5]
}

random_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        random_state=42
    ),
    param_distributions=params,
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV Score:")
print(random_search.best_score_)
```

---

# 19. What Does `n_iter` Mean?

You previously had confusion about this with Gradient Boosting.

In:

```python
RandomizedSearchCV(
    ...,
    n_iter=30
)
```

`n_iter=30` means:

> Randomly select and evaluate **30 hyperparameter combinations**.

It does **not** mean 30 trees.

These are completely different:

```python
n_estimators=300
```

means:

> Up to 300 boosting trees.

Whereas:

```python
n_iter=30
```

means:

> Test 30 randomly selected hyperparameter combinations.

---

# 20. RandomizedSearchCV + CV

If:

```python
n_iter=30
```

and:

```python
cv=5
```

then approximately:

$$
30\times5=150
$$

fits are performed.

This is much cheaper than searching hundreds or thousands of combinations.

---

# 21. Which Parameters Should You Tune?

A useful parameter search:

```python
params = {

    "n_estimators":
        [100, 200, 300, 500],

    "learning_rate":
        [0.01, 0.03, 0.05, 0.1],

    "max_depth":
        [3, 4, 5, 6, 8],

    "min_child_weight":
        [1, 3, 5, 7],

    "gamma":
        [0, 0.1, 0.3, 0.5],

    "subsample":
        [0.6, 0.8, 1.0],

    "colsample_bytree":
        [0.6, 0.8, 1.0],

    "reg_alpha":
        [0, 0.01, 0.1, 1],

    "reg_lambda":
        [1, 2, 5, 10]
}
```

---

# 22. Don't Tune Everything at Once

This is a very important practical lesson.

Don't immediately throw 20 parameters into GridSearch.

Instead:

### Phase 1 — Tree complexity

```text
max_depth
min_child_weight
gamma
```

### Phase 2 — Sampling

```text
subsample
colsample_bytree
```

### Phase 3 — Regularization

```text
reg_alpha
reg_lambda
```

### Phase 4 — Boosting

```text
learning_rate
n_estimators
```

This makes the tuning process easier to understand.

---

# 23. Early Stopping

This is one of the most useful features of XGBoost.

Suppose:

```python
n_estimators=1000
```

You don't necessarily want all 1000 trees.

Maybe performance improves like this:

```text
Tree       Validation Score

10         0.70
50         0.78
100        0.82
200        0.84
300        0.85
400        0.85
500        0.84
600        0.83
```

The model has started getting worse.

That's overfitting.

Early stopping allows training to stop when validation performance stops improving.

---

# 24. Early Stopping Concept

```text
Training
   ↓
Tree 1
   ↓
Tree 2
   ↓
Tree 3
   ↓
...
   ↓
Validation improves
   ↓
Validation improves
   ↓
Validation stops improving
   ↓
Wait for patience period
   ↓
STOP
```

---

# 25. Validation Set for Early Stopping

You should ideally have:

```text
Training set
Validation set
Test set
```

Example:

```python
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)
```

So:

```text
70% → Training
15% → Validation
15% → Test
```

---

# 26. Early Stopping Example

XGBoost's sklearn-style API has changed across versions, so use the early-stopping interface supported by the version you have installed.

For current versions, a common pattern is:

```python
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    random_state=42,
    eval_metric="rmse",
    early_stopping_rounds=50
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
```

Here:

```python
early_stopping_rounds=50
```

means that if the validation metric doesn't improve for 50 consecutive boosting rounds, training stops.

---

# 27. Finding the Best Number of Trees

After early stopping:

```python
print(model.best_iteration)
```

Depending on the XGBoost version/API, you can also inspect the best score:

```python
print(model.best_score)
```

This tells you approximately where the validation performance was best.

---

# 28. Why Early Stopping Is Better Than Guessing `n_estimators`

Instead of:

```python
n_estimators=100
```

and hoping it's enough,

you can use:

```python
n_estimators=1000
```

with early stopping.

The model has room to learn, but training can stop when additional trees stop improving validation performance.

Conceptually:

$$
\boxed{
Large\ n\_estimators + Early\ Stopping
}
$$

can be a very useful strategy.

---

# 29. Important Warning About Test Data

Don't do this:

```python
eval_set=[(X_test, y_test)]
```

if you're going to use the test set for your final evaluation.

Why?

Because you're allowing the training procedure to make decisions based on your test data.

Better:

```text
Train → learning
Validation → early stopping / tuning
Test → final evaluation
```

---

# 30. Feature Importance

After training:

```python
model.feature_importances_
```

Example:

```python
importance = model.feature_importances_

print(importance)
```

If you have a DataFrame:

```python
import pandas as pd

feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

print(
    feature_importance
    .sort_values(ascending=False)
)
```

---

# 31. Plot Feature Importance

```python
import matplotlib.pyplot as plt

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

importance.sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance")
plt.title("XGBoost Feature Importance")
plt.show()
```

---

# 32. What Does Feature Importance Mean?

Suppose you get:

```text
MedInc       0.35
AveRooms     0.18
HouseAge     0.15
AveOccup     0.10
Population   0.08
...
```

This suggests:

> `MedInc` contributed strongly to the model's tree-based decision process.

But **feature importance does not automatically mean causation**.

For example:

```text
Important feature
       ≠
Cause of target
```

It only describes the model's use of that feature.

---

# 33. Gain vs Weight vs Cover

XGBoost can calculate feature importance using different concepts.

### Weight

How often the feature is used in splits.

### Gain

How much the feature improves the objective when used for splitting.

### Cover

How many observations are affected by those splits.

For interpretability, **gain-based importance** is often more informative than simply counting how frequently a feature appears.

---

# 34. Getting Gain-Based Importance

Using XGBoost's underlying booster:

```python
booster = model.get_booster()

importance = booster.get_score(
    importance_type="gain"
)

print(importance)
```

Other options include:

```python
importance_type="weight"
```

and:

```python
importance_type="cover"
```

---

# 35. XGBoost Built-in Plot

You can also use:

```python
from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(
    model,
    importance_type="gain"
)

plt.show()
```

This is useful for quickly inspecting the model.

---

# 36. Important Limitation of Feature Importance

Suppose:

```text
Feature A → highly correlated with Feature B
```

The model may use A heavily and B very little.

That does **not necessarily mean B is unimportant to the underlying problem**.

It may simply mean that A already provides similar information.

Therefore, for deeper model interpretation, you can later use:

```text
PDP
SHAP
Permutation Importance
```

This connects directly to the PDP work you've already started.

---

# 37. XGBoost End-to-End Regression Workflow

Here's the workflow you should practice:

```text
Dataset
   ↓
EDA
   ↓
Train / Validation / Test
   ↓
Baseline XGBRegressor
   ↓
Evaluate
   ↓
RandomizedSearchCV
   ↓
Best Hyperparameters
   ↓
Train Tuned Model
   ↓
Early Stopping
   ↓
Evaluate on Test
   ↓
Feature Importance
   ↓
PDP
   ↓
SHAP
```

---

# 38. Complete Practical Code

Here's a clean version you can actually use for your California Housing experiment.

```python
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from xgboost import XGBRegressor


# -----------------------------
# 1. Load Data
# -----------------------------

data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target


# -----------------------------
# 2. Train / Validation / Test
# -----------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


# -----------------------------
# 3. Create Model
# -----------------------------

model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=1,
    early_stopping_rounds=50,
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)


# -----------------------------
# 4. Train
# -----------------------------

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)


# -----------------------------
# 5. Prediction
# -----------------------------

y_pred = model.predict(X_test)


# -----------------------------
# 6. Evaluation
# -----------------------------

r2 = r2_score(y_test, y_pred)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

print("R²:", r2)
print("MSE:", mse)
print("RMSE:", rmse)


# -----------------------------
# 7. Best Iteration
# -----------------------------

print(
    "Best Iteration:",
    model.best_iteration
)


# -----------------------------
# 8. Feature Importance
# -----------------------------

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

print(
    importance
    .sort_values(ascending=False)
)
```

---

# 39. XGBClassifier End-to-End Workflow

For classification:

```python
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=1,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=1,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]
```

Evaluation:

```python
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1:",
    f1_score(y_test, y_pred)
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_prob)
)
```

---

# 40. What You Should Memorize

### Classifier

```python
XGBClassifier()
```

for classification.

### Regressor

```python
XGBRegressor()
```

for regression.

### Main parameters

```text
n_estimators
learning_rate
max_depth
min_child_weight
gamma
subsample
colsample_bytree
reg_alpha
reg_lambda
```

### Classification probability

```python
predict_proba()
```

### Regression prediction

```python
predict()
```

### Hyperparameter tuning

```python
GridSearchCV
RandomizedSearchCV
```

### Early stopping

```python
early_stopping_rounds
```

### Feature importance

```python
feature_importances_
```

or:

```python
get_booster().get_score()
```

---

# 41. XGBoost vs Your Previous Models

Since you're learning these algorithms sequentially, this is the comparison you should keep in your notes:

| Model             | Main Idea                     | Sequential? | Regularization |         Uses Gradient? |
| ----------------- | ----------------------------- | ----------: | -------------: | ---------------------: |
| Decision Tree     | Single tree                   |           ❌ |        Limited |                      ❌ |
| Random Forest     | Bagging                       |           ❌ |       Indirect |                      ❌ |
| AdaBoost          | Focus on difficult samples    |           ✅ |        Limited |           Not directly |
| Gradient Boosting | Fit negative gradients        |           ✅ |           Some |                      ✅ |
| **XGBoost**       | Regularized gradient boosting |           ✅ |     **Strong** | **Gradient + Hessian** |

The progression you've learned is therefore:

```text
Decision Tree
      ↓
Bagging
      ↓
Random Forest
      ↓
Boosting
      ↓
AdaBoost
      ↓
Gradient Boosting
      ↓
XGBoost
```

And the central evolution is:

```text
Decision Tree
    ↓
Many Trees
    ↓
Sequential Trees
    ↓
Gradient-based corrections
    ↓
Gradient + Hessian
    ↓
Regularization + sampling + optimization
    ↓
XGBoost
```

### The next logical step

After these Parts 8–13, the next section should be **Part 14–15: XGBoost Feature Interpretation using PDP + SHAP**, followed by a **complete XGBoost project** where you compare **Decision Tree vs Random Forest vs AdaBoost vs Gradient Boosting vs XGBoost** on the same dataset.
